# YOLO → SAM 균열 세그멘테이션 최종 프로세스

이 노트북은 **YOLO 균열 마스크를 위치 힌트로 사용하고, SAM이 경계를 보정**하도록 구성되어 있습니다.

- 한 장 또는 여러 장을 순차 처리합니다.
- SAM 결과가 YOLO 위치에서 과도하게 벗어나지 않도록 공간 제한을 적용합니다.
- 작은 오탐 영역을 제거합니다.
- 품질 조건을 통과하지 못한 SAM 결과는 YOLO 마스크로 되돌립니다.
- 검수용 PNG와 Roboflow용 YOLO polygon ZIP을 각각 만듭니다.

> Colab 메뉴에서 `런타임 → 런타임 유형 변경 → T4 GPU`를 먼저 선택한 뒤, 위에서 아래로 셀을 실행하세요.

## 1. 라이브러리 설치

In [ ]:
!pip -q install -U ultralytics opencv-python-headless matplotlib pandas tqdm

## 2. Google Drive 연결

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. 설정

`YOLO_MODEL_PATH`와 `INPUT_DIR`만 본인 경로에 맞게 확인하세요. 입력 폴더에는 원본 사진만 넣습니다.

In [ ]:
from pathlib import Path

YOLO_MODEL_PATH = Path('/content/drive/MyDrive/YOLO11_Crack_Epoch_Comparison/yolo11s_seg_100epochs-2/weights/best.pt')
SAM_MODEL_NAME = 'sam2.1_s.pt'
INPUT_DIR = Path('/content/drive/MyDrive/YOLO_SAM_Input')
OUTPUT_DIR = Path('/content/drive/MyDrive/YOLO_SAM_Final_Results')

# 추론 및 후처리 설정
YOLO_CONF = 0.25
YOLO_IMGSZ = 1280
MIN_YOLO_AREA = 30          # 너무 작은 YOLO 인스턴스 무시
MIN_COMPONENT_AREA = 100    # 최종 마스크의 작은 조각 제거
DILATION_SIZE = 21          # SAM이 YOLO 주변에서 움직일 수 있는 범위(홀수)
MIN_IOU = 0.50              # SAM-YOLO 최소 IoU
MIN_AREA_RATIO = 0.50       # SAM 면적 / YOLO 면적 하한
MAX_AREA_RATIO = 2.00       # SAM 면적 / YOLO 면적 상한
POLYGON_EPSILON_RATIO = 0.001
MIN_EXPORT_AREA = 100

INPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('YOLO 모델:', YOLO_MODEL_PATH)
print('모델 존재:', YOLO_MODEL_PATH.exists())
print('입력 폴더:', INPUT_DIR)
print('결과 폴더:', OUTPUT_DIR)
assert YOLO_MODEL_PATH.exists(), 'YOLO_MODEL_PATH를 실제 best.pt 경로로 수정하세요.'

## 4. 입력 사진 확인

Google Drive의 `YOLO_SAM_Input` 폴더에 JPG/PNG 사진을 넣은 뒤 실행하세요. 사진을 많이 넣어도 되며, 한 장씩 순차 처리됩니다.

In [ ]:
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}
image_paths = sorted([p for p in INPUT_DIR.rglob('*') if p.suffix.lower() in IMAGE_EXTENSIONS])

print(f'발견된 사진: {len(image_paths)}장')
for p in image_paths[:10]:
    print('-', p.name)
if len(image_paths) > 10:
    print(f'... 외 {len(image_paths) - 10}장')
assert image_paths, f'{INPUT_DIR} 폴더에 사진을 넣으세요.'

## 5. 모델 및 공통 모듈 불러오기

In [ ]:
import gc
import json
import shutil
import cv2
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from ultralytics import YOLO
from ultralytics.models.sam import SAM2Predictor

DEVICE = 0 if torch.cuda.is_available() else 'cpu'
print('사용 장치:', DEVICE)

yolo_model = YOLO(str(YOLO_MODEL_PATH))
sam_predictor = SAM2Predictor(overrides={
    'model': SAM_MODEL_NAME,
    'task': 'segment',
    'mode': 'predict',
    'imgsz': 1024,
    'conf': 0.25,
    'device': DEVICE,
})
print('모델 준비 완료')

## 6. 후처리·저장 함수

아래 셀은 작은 조각 제거, 품질 검사, 색상 비교 이미지, YOLO polygon 변환을 담당합니다.

In [ ]:
def remove_small_components(mask, min_area=100):
    mask_u8 = (mask > 0).astype(np.uint8)
    count, labels, stats, _ = cv2.connectedComponentsWithStats(mask_u8, connectivity=8)
    cleaned = np.zeros_like(mask_u8)
    for label in range(1, count):
        if stats[label, cv2.CC_STAT_AREA] >= min_area:
            cleaned[labels == label] = 1
    return cleaned.astype(bool)


def mask_iou(mask_a, mask_b):
    intersection = np.logical_and(mask_a, mask_b).sum()
    union = np.logical_or(mask_a, mask_b).sum()
    return float(intersection / union) if union else 0.0


def make_comparison_overlay(image_rgb, yolo_mask, sam_mask, alpha=0.55):
    # 빨강: YOLO만, 초록: SAM만, 노랑: 공통 영역
    overlay = image_rgb.astype(np.float32).copy()
    yolo_only = np.logical_and(yolo_mask, ~sam_mask)
    sam_only = np.logical_and(sam_mask, ~yolo_mask)
    overlap = np.logical_and(yolo_mask, sam_mask)
    colors = [
        (yolo_only, np.array([255, 40, 40], dtype=np.float32)),
        (sam_only, np.array([30, 255, 90], dtype=np.float32)),
        (overlap, np.array([255, 220, 20], dtype=np.float32)),
    ]
    for region, color in colors:
        overlay[region] = overlay[region] * (1 - alpha) + color * alpha
    return np.clip(overlay, 0, 255).astype(np.uint8)


def mask_to_yolo_lines(mask, class_id=0, epsilon_ratio=0.001, min_area=100):
    h, w = mask.shape
    contours, _ = cv2.findContours(
        (mask.astype(np.uint8) * 255), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE
    )
    lines = []
    for contour in contours:
        if cv2.contourArea(contour) < min_area:
            continue
        epsilon = epsilon_ratio * cv2.arcLength(contour, True)
        polygon = cv2.approxPolyDP(contour, epsilon, True).reshape(-1, 2)
        if len(polygon) < 3:
            continue
        coords = []
        for x, y in polygon:
            coords.extend([np.clip(x / w, 0, 1), np.clip(y / h, 0, 1)])
        lines.append(str(class_id) + ' ' + ' '.join(f'{v:.6f}' for v in coords))
    return lines

## 7. SAM 보정 함수

중요: 고수준 `predict(masks=...)` 호출은 Ultralytics 버전에 따라 오류가 날 수 있습니다. 여기서는 YOLO 마스크를 SAM의 256×256 dense prompt로 직접 전달해 기존의 `64 vs 256` 크기 오류를 피합니다.

In [ ]:
def refine_with_sam(yolo_mask, original_hw):
    h, w = original_hw

    # SAM prompt encoder가 요구하는 dense mask 크기
    prompt_256 = cv2.resize(
        yolo_mask.astype(np.uint8), (256, 256), interpolation=cv2.INTER_NEAREST
    )
    # 이진값 대신 강한 foreground/background logit 사용
    prompt_logits = np.where(prompt_256 > 0, 10.0, -10.0).astype(np.float32)
    prompt_tensor = torch.from_numpy(prompt_logits)[None, None].to(sam_predictor.device)

    with torch.inference_mode():
        output = sam_predictor._inference_features(
            features=sam_predictor.features,
            masks=prompt_tensor,
            multimask_output=False,
        )

    pred = output[0] if isinstance(output, (tuple, list)) else output
    if isinstance(pred, torch.Tensor):
        pred = pred.detach().float().cpu().numpy()
    pred = np.asarray(pred).squeeze()
    if pred.ndim != 2:
        raise RuntimeError(f'예상하지 못한 SAM 출력 크기: {pred.shape}')

    # logits이면 0, 확률/이진값이면 0.5를 기준으로 이진화
    threshold = 0.0 if pred.min() < 0 else 0.5
    raw_mask = pred > threshold
    raw_mask = cv2.resize(raw_mask.astype(np.uint8), (w, h), interpolation=cv2.INTER_NEAREST).astype(bool)
    return raw_mask

## 8. 사진 한 장 처리 함수

In [ ]:
MASK_YOLO_DIR = OUTPUT_DIR / 'masks' / 'yolo'
MASK_SAM_RAW_DIR = OUTPUT_DIR / 'masks' / 'sam_raw'
MASK_FINAL_DIR = OUTPUT_DIR / 'masks' / 'final'
OVERLAY_DIR = OUTPUT_DIR / 'comparison_overlays'
RF_IMAGE_DIR = OUTPUT_DIR / 'roboflow_dataset' / 'images' / 'train'
RF_LABEL_DIR = OUTPUT_DIR / 'roboflow_dataset' / 'labels' / 'train'

for folder in [MASK_YOLO_DIR, MASK_SAM_RAW_DIR, MASK_FINAL_DIR, OVERLAY_DIR, RF_IMAGE_DIR, RF_LABEL_DIR]:
    folder.mkdir(parents=True, exist_ok=True)


def process_one_image(image_path):
    image_path = Path(image_path)
    image_bgr = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
    if image_bgr is None:
        raise RuntimeError(f'이미지를 읽지 못했습니다: {image_path}')
    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    h, w = image_rgb.shape[:2]
    stem = image_path.stem

    # 원본을 Roboflow 검수 폴더로 복사
    export_image_path = RF_IMAGE_DIR / image_path.name
    shutil.copy2(image_path, export_image_path)

    yolo_result = yolo_model.predict(
        source=str(image_path), conf=YOLO_CONF, imgsz=YOLO_IMGSZ,
        device=DEVICE, retina_masks=True, verbose=False
    )[0]

    yolo_union = np.zeros((h, w), dtype=bool)
    sam_raw_union = np.zeros((h, w), dtype=bool)
    final_union = np.zeros((h, w), dtype=bool)
    instance_records = []

    if yolo_result.masks is not None and len(yolo_result.masks.data) > 0:
        # SAM은 이미지 특징을 한 번만 계산하고 여러 YOLO 인스턴스에 재사용
        sam_predictor.reset_image()
        sam_predictor.set_image(str(image_path))

        for instance_id, mask_tensor in enumerate(yolo_result.masks.data):
            yolo_mask = mask_tensor.detach().cpu().numpy()
            yolo_mask = cv2.resize(yolo_mask, (w, h), interpolation=cv2.INTER_NEAREST) > 0.5
            yolo_area = int(yolo_mask.sum())
            if yolo_area < MIN_YOLO_AREA:
                continue

            raw_sam = refine_with_sam(yolo_mask, (h, w))

            # SAM이 균열 주변의 콘크리트 전체를 선택하지 못하도록 공간 제한
            kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (DILATION_SIZE, DILATION_SIZE))
            allowed = cv2.dilate(yolo_mask.astype(np.uint8), kernel, iterations=1).astype(bool)
            constrained = np.logical_and(raw_sam, allowed)
            constrained = remove_small_components(constrained, MIN_COMPONENT_AREA)

            iou = mask_iou(yolo_mask, constrained)
            area_ratio = float(constrained.sum() / max(yolo_area, 1))
            accepted = bool(
                constrained.any() and
                iou >= MIN_IOU and
                MIN_AREA_RATIO <= area_ratio <= MAX_AREA_RATIO
            )
            chosen = constrained if accepted else yolo_mask

            yolo_union |= yolo_mask
            sam_raw_union |= raw_sam
            final_union |= chosen
            instance_records.append({
                'instance_id': instance_id,
                'yolo_pixels': yolo_area,
                'sam_pixels': int(constrained.sum()),
                'iou': iou,
                'area_ratio': area_ratio,
                'sam_accepted': accepted,
            })

    final_union = remove_small_components(final_union, MIN_COMPONENT_AREA)

    # 마스크와 비교 이미지 저장
    cv2.imwrite(str(MASK_YOLO_DIR / f'{stem}.png'), yolo_union.astype(np.uint8) * 255)
    cv2.imwrite(str(MASK_SAM_RAW_DIR / f'{stem}.png'), sam_raw_union.astype(np.uint8) * 255)
    cv2.imwrite(str(MASK_FINAL_DIR / f'{stem}.png'), final_union.astype(np.uint8) * 255)
    overlay_rgb = make_comparison_overlay(image_rgb, yolo_union, final_union)
    cv2.imwrite(str(OVERLAY_DIR / f'{stem}.jpg'), cv2.cvtColor(overlay_rgb, cv2.COLOR_RGB2BGR))

    # Roboflow용 YOLO segmentation polygon TXT
    label_lines = mask_to_yolo_lines(
        final_union, class_id=0, epsilon_ratio=POLYGON_EPSILON_RATIO, min_area=MIN_EXPORT_AREA
    )
    (RF_LABEL_DIR / f'{stem}.txt').write_text('\n'.join(label_lines), encoding='utf-8')

    accepted_count = sum(r['sam_accepted'] for r in instance_records)
    return {
        'image': image_path.name,
        'width': w,
        'height': h,
        'yolo_instances': len(instance_records),
        'sam_accepted': accepted_count,
        'fallback_to_yolo': len(instance_records) - accepted_count,
        'yolo_pixels': int(yolo_union.sum()),
        'final_pixels': int(final_union.sum()),
        'polygon_count': len(label_lines),
        'status': 'ok' if instance_records else 'no_detection',
        'instances': instance_records,
    }

## 9. 전체 사진 일괄 처리

사진 수가 많으면 시간이 오래 걸릴 수 있지만 한 장씩 순차 처리하므로 GPU 메모리 부담을 줄일 수 있습니다.

In [ ]:
summaries = []
errors = []

for image_path in tqdm(image_paths, desc='YOLO → SAM 처리'):
    try:
        summaries.append(process_one_image(image_path))
    except Exception as exc:
        errors.append({'image': image_path.name, 'error': repr(exc)})
        print(f'오류 - {image_path.name}: {exc}')
    finally:
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

summary_df = pd.DataFrame([{k: v for k, v in row.items() if k != 'instances'} for row in summaries])
error_df = pd.DataFrame(errors)
summary_df.to_csv(OUTPUT_DIR / 'processing_summary.csv', index=False, encoding='utf-8-sig')
error_df.to_csv(OUTPUT_DIR / 'processing_errors.csv', index=False, encoding='utf-8-sig')
(OUTPUT_DIR / 'instance_details.json').write_text(
    json.dumps(summaries, ensure_ascii=False, indent=2), encoding='utf-8'
)

print(f'완료: {len(summaries)}장 / 오류: {len(errors)}장')
display(summary_df)
if not error_df.empty:
    display(error_df)

## 10. 결과 미리보기

In [ ]:
preview_paths = sorted(OVERLAY_DIR.glob('*.jpg'))[:6]
if preview_paths:
    cols = 2
    rows = int(np.ceil(len(preview_paths) / cols))
    plt.figure(figsize=(14, 7 * rows))
    for index, preview_path in enumerate(preview_paths, 1):
        preview = cv2.cvtColor(cv2.imread(str(preview_path)), cv2.COLOR_BGR2RGB)
        plt.subplot(rows, cols, index)
        plt.imshow(preview)
        plt.title(preview_path.name)
        plt.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print('미리보기 이미지가 없습니다.')

print('색상 의미: 빨강=YOLO만, 초록=SAM만, 노랑=공통 영역')

## 11. Roboflow 업로드용 ZIP 만들기

Roboflow ZIP에는 **원본 이미지 + YOLO segmentation TXT + data.yaml**만 포함합니다. PNG 마스크를 함께 넣으면 `segmentation mask files are not yet supported` 오류가 날 수 있습니다.

In [ ]:
ROBOFLOW_ROOT = OUTPUT_DIR / 'roboflow_dataset'
data_yaml = """path: .
train: images/train
val: images/train
names:
  0: crack
"""
(ROBOFLOW_ROOT / 'data.yaml').write_text(data_yaml, encoding='utf-8')

zip_base = OUTPUT_DIR / 'YOLO_SAM_Roboflow_Review'
zip_path = shutil.make_archive(str(zip_base), 'zip', root_dir=ROBOFLOW_ROOT)
print('Roboflow ZIP 저장 완료:', zip_path)
print('검수용 PNG 결과:', OUTPUT_DIR / 'comparison_overlays')
print('최종 이진 마스크:', OUTPUT_DIR / 'masks' / 'final')

## 12. 필요하면 ZIP을 내 컴퓨터로 다운로드

In [ ]:
from google.colab import files
files.download(zip_path)

## 결과 폴더 구조

- `masks/yolo`: YOLO 원본 이진 마스크
- `masks/sam_raw`: 공간 제한 전 SAM 마스크
- `masks/final`: 제한·작은 조각 제거·품질 검사를 통과한 최종 마스크
- `comparison_overlays`: 사람이 비교 검수할 색상 이미지
- `roboflow_dataset`: Roboflow에서 polygon을 직접 수정할 데이터
- `processing_summary.csv`: 이미지별 처리 요약
- `processing_errors.csv`: 실패한 이미지와 오류 내용

처음에는 10~20장으로 시험한 뒤 `DILATION_SIZE`, `MIN_COMPONENT_AREA`, `MIN_IOU`를 조정하고 전체 사진을 처리하는 것을 권장합니다.